In [32]:
#1 — Imports
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional
import os
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_google_genai import GoogleGenerativeAIEmbeddings


In [33]:
#2 — Load Environment
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

assert GEMINI_API_KEY
assert PINECONE_API_KEY

In [34]:
#3 — Configuration Class
@dataclass(slots=True)
class RetrieverConfig:
    index_name:str
    embedding_model:str
    embedding_dimension:int
    top_k:int=5


In [35]:
config = RetrieverConfig(
    index_name="supportsphere-ai",
    embedding_model="models/gemini-embedding-001",
    embedding_dimension=768,
    top_k=5,
)

In [36]:
#4 — Build Retriever
class PineconeRetriever:

    def __init__(self, config: RetrieverConfig):
        self.config = config
        self.pc = Pinecone(api_key=PINECONE_API_KEY)
        self.index = self.pc.Index(config.index_name)
        self.embedding_model = (GoogleGenerativeAIEmbeddings(
                                model=config.embedding_model,
                                output_dimensionality=config.embedding_dimension))
        
    
    #Retrieve Method
    def retrieve(self, question: str, company: Optional[str] = None, 
             product_area: Optional[str] = None,top_k: Optional[int] = None,):

        query_vector = self.embedding_model.embed_query(question)
        filters = {}
        if company:
            filters["company"] = company.lower()

        if product_area:
            filters["product_area"] = product_area

        results = self.index.query(vector=query_vector,
                                    top_k=top_k or self.config.top_k,
                                    include_metadata=True,
                                    filter=filters if filters else None,)

        return results.matches
    
    #Build Context
    @staticmethod
    def build_context(matches):
        return "\n\n".join(match["metadata"]["text"] for match in matches)
    
    #Pretty Printer
    @staticmethod
    def print_matches(matches):

        for i, match in enumerate(matches, start=1):
            print("="*80)
            print(f"Rank : {i}")
            print(f"Score : {match['score']:.4f}")
            print()
            print("Company :", match["metadata"]["company"])
            print("Product :", match["metadata"]["product_area"])
            print("Title :", match["metadata"]["title"])
            print()
            print(match["metadata"]["text"][:500])


In [37]:
#5 — Initialize
retriever = PineconeRetriever(config)

In [38]:
#6 — First Query
matches = retriever.retrieve(

    question="How do I reset my Claude password?"

)

retriever.print_matches(matches)

Rank : 1
Score : 0.7820

Company : claude
Product : claude
Title : "Logging in to your Claude account"

How can I set a password for my Claude account?
It’s not possible to create a dedicated password for your Claude account at this time.
Can I use my Claude account across multiple devices?
To use your Claude account across multiple devices, enter the same email address you use to log in on your usual device. For example, if you signed up for your Claude account on a web browser, you can access the same account on the Claude mobile apps by entering the same email address when you log in.
I have mu
Rank : 2
Score : 0.7722

Company : claude
Product : claude
Title : "Logging in to your Claude account"

**Note: **Business, government, or .edu email users may need to contact their IT department to adjust email security settings.
I received the email but I'm still having trouble logging in.
If you received the login email but can’t log in with the link or code, take note of the error message

In [39]:
#7 — Company Filter
matches = retriever.retrieve(

    question="Reset password",

    company="hackerrank",

)

retriever.print_matches(matches)

Rank : 1
Score : 0.7056

Company : hackerrank
Product : settings
Title : "Update or Reset Password Updating password Resetting password"

Resetting password
To reset a forgotten password:
Go to the HackerRank for Work login page.
Click **Forgot Password?** under the password field.
 <img src="https://assets.usepylon.com/e6a58e21-be80-4777-9eaf-f73beeee94d9%2Fba2098a0-25fd-4540-be8b-22162546efef-AD_4nXcfZaFqveIjoF0r78h6u7fu_b1gf2jWGjsUpZbCwCr6rdvYuke6rpzZhzI1LPo2LBGmjEw1KnssNTIdqkhvoOyjKrE2mZbRfm9y4GLDdSQ7BfFhOhD7Kjxu_4Is6fGgzGRrcN6IjQ-4d4309e9-1afe-454b-930a-df09793c8234?Expires=253370764800&amp;Signature=VRYahrGPT-g55WR8P1-HyUiYY
Rank : 2
Score : 0.7032

Company : hackerrank
Product : hackerrank_community
Title : "Update or Reset Password"

Resetting password
To reset a forgotten password:
Go to the HackerRank Community login page.
Click **Forgot Password?** under the password field.
Enter the email address or username associated with your account and select the verification checkbox.

In [40]:
#8 — Product Area Filter
matches = retriever.retrieve(

    question="How do I access Claude?",

    company="claude",

    product_area="amazon_bedrock",
)
retriever.print_matches(matches)

Rank : 1
Score : 0.7814

Company : claude
Product : amazon_bedrock
Title : "How do I get access to Claude in Amazon Bedrock?"

How do I get access to Claude in Amazon Bedrock?
_Last updated: 2026-03-16T21:16:50Z_
Get started with Claude in Amazon Bedrock by visiting the Amazon Bedrock console. For a step-by-step walkthrough on how to request Claude model access in the Amazon Bedrock console, view this blog.
Related Articles
What is Amazon Bedrock?
I use Claude in Amazon Bedrock. Who do I contact for customer support inquiries?
Where do I find Claude in Amazon Bedrock documentation?
What AWS Regions are Claude models ava
Rank : 2
Score : 0.7401

Company : claude
Product : amazon_bedrock
Title : "What is Amazon Bedrock?"

You can learn more about Anthropic’s Claude models in Amazon Bedrock here.
Related Articles
How do I get access to Claude in Amazon Bedrock?
I use Claude in Amazon Bedrock. Who do I contact for customer support inquiries?
What AWS Regions are Claude models available in 

In [41]:
#9 — Build Context
context = retriever.build_context(matches)

print(context[:3000])

How do I get access to Claude in Amazon Bedrock?
_Last updated: 2026-03-16T21:16:50Z_
Get started with Claude in Amazon Bedrock by visiting the Amazon Bedrock console. For a step-by-step walkthrough on how to request Claude model access in the Amazon Bedrock console, view this blog.
Related Articles
What is Amazon Bedrock?
I use Claude in Amazon Bedrock. Who do I contact for customer support inquiries?
Where do I find Claude in Amazon Bedrock documentation?
What AWS Regions are Claude models available in Amazon Bedrock?
Claude Code FAQ

You can learn more about Anthropic’s Claude models in Amazon Bedrock here.
Related Articles
How do I get access to Claude in Amazon Bedrock?
I use Claude in Amazon Bedrock. Who do I contact for customer support inquiries?
What AWS Regions are Claude models available in Amazon Bedrock?
Public Sector FAQs
Use Claude for Excel, PowerPoint, and Word with third-party platforms

Related Articles
What is Amazon Bedrock?
What AWS Regions are Claude models avail

In [42]:
#10 — Notebook Summary
print("=" * 60)

print("PRODUCTION RETRIEVER COMPLETE")

print("=" * 60)

print("Retriever Class ✓")

print("Semantic Search ✓")

print("Metadata Filtering ✓")

print("Context Builder ✓")

PRODUCTION RETRIEVER COMPLETE
Retriever Class ✓
Semantic Search ✓
Metadata Filtering ✓
Context Builder ✓


                    LangGraph
                         │
                         ▼
                BaseRetriever (ABC)
                         │
        ┌────────────────┴────────────────┐
        │                                 │
        ▼                                 ▼
FAISSRetriever                 PineconeRetriever
        │                                 │
        ▼                                 ▼
     FAISS                         Pinecone Cloud

     

                    BaseRetriever
                          │
      ┌──────────────────┬───────────────────────────┐
      ▼                  ▼                           ▼
FAISSRetriever      |    PineconeRetriever       |QdrantRetriever



## Project Structure (Phase 2)

In [43]:
#1 — Imports
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import os
import pickle

import faiss
import numpy as np

from dotenv import load_dotenv

from pinecone import Pinecone

from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [44]:
#2 — Environment
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [45]:
#3 — Config
@dataclass(slots=True)
class RetrieverConfig:

    embedding_model: str

    embedding_dimension: int

    top_k: int = 5

In [46]:
#4 — BaseRetriever
class BaseRetriever(ABC):

    @abstractmethod
    def retrieve(self, question: str, company: Optional[str] = None,
                product_area: Optional[str] = None, top_k: Optional[int] = None,):
        """Return retrieved matches."""
        pass

    @abstractmethod
    def build_context(self, matches,) -> str:
        """Build LLM context."""
        pass

In [47]:
#5 — BaseVectorRetriever
class BaseVectorRetriever(BaseRetriever):

    def __init__(self, config: RetrieverConfig,):
        self.config = config
        self.embedding_model = GoogleGenerativeAIEmbeddings(
                                model=config.embedding_model,
                                output_dimensionality=config.embedding_dimension,)
    

    #Shared embedding function
    def embed_query(self, question: str,):
        return self.embedding_model.embed_query(question)


    #Shared context builder
    def build_context(self, matches,):
        texts = []
        for match in matches:
            if isinstance(match, dict):
                texts.append(match["metadata"]["text"])
            else:
                texts.append(match.metadata["text"])

        return "\n\n".join(texts)    


    #Shared pretty printer
    def print_matches(self, matches,):

        for i, match in enumerate(matches, start=1):
            print("=" * 80)
            print(f"Rank : {i}")

            if isinstance(match, dict):
                metadata = match["metadata"]
                score = match.get("score")

            else:
                metadata = match.metadata
                score = match.score

            print(f"Score : {score:.4f}")
            print()
            print("Company :", metadata["company"])
            print("Product :", metadata["product_area"])
            print("Title :", metadata["title"])
            print()
            print(metadata["text"][:500])    

In [48]:
#6 — FAISSRetriever
class FAISSRetriever(BaseVectorRetriever):

    def __init__(self, config, index_path, metadata_path,):

        super().__init__(config)

        self.index = faiss.read_index(index_path)

        with open(metadata_path, "rb") as f:

            self.metadata = pickle.load(f)

    def retrieve(self, question, company=None, product_area=None, top_k=None,):

        query = self.embed_query(question)

        query = np.array([query], dtype=np.float32,)

        distances, indices = self.index.search(query, top_k or self.config.top_k,)

        matches = []

        for distance, idx in zip(distances[0],indices[0],):

            doc = self.metadata[idx]

            if company:

                if doc["metadata"]["company"] != company.lower():
                    continue

            if product_area:

                if doc["metadata"]["product_area"] != product_area:
                    continue

            matches.append({

                "score": float(distance),

                "metadata": doc["metadata"],
            })

        return matches

In [49]:
#7 — PineconeRetriever
class PineconeRetriever(BaseVectorRetriever):

    def __init__(self, config, index_name,):
        super().__init__(config)
        self.pc = Pinecone(api_key=PINECONE_API_KEY)
        self.index = self.pc.Index(index_name)
    
    def retrieve(self, question, company=None, product_area=None, top_k=None,):

        query = self.embed_query(question)
        filters = {}

        if company:
            filters["company"] = company.lower()

        if product_area:
            filters["product_area"] = product_area

        response = self.index.query(vector=query,
                                    top_k=top_k or self.config.top_k,
                                    include_metadata=True,
                                    filter=filters if filters else None,)

        matches = []

        for match in response.matches:
            matches.append({"score": float(match.score), "metadata": dict(match.metadata),})

        return matches    

In [50]:
#8 — Configuration
config = RetrieverConfig(

    embedding_model="models/gemini-embedding-001",

    embedding_dimension=768,

    top_k=5,
)

In [51]:
#9 — Create FAISS Retriever
faiss_retriever = FAISSRetriever(

    config=config,

    index_path="../artifacts/faiss_index.index",

    metadata_path="../artifacts/faiss_metadata.pkl",
)

In [52]:
#10 — Create Pinecone Retriever
pinecone_retriever = PineconeRetriever(

    config=config,

    index_name="supportsphere-ai",
)

In [59]:
#11 — Test FAISS
matches = faiss_retriever.retrieve(
    question="How do I reset my password?",
)

len(matches)

5

In [ ]:
for m in matches:
    print(m["metadata"]["company"])

hackerrank
hackerrank
hackerrank
hackerrank
hackerrank


In [56]:
#12 — Test Pinecone
matches = pinecone_retriever.retrieve(

    question="How do I reset my password?",

    company="claude",
)

pinecone_retriever.print_matches(matches)

Rank : 1
Score : 0.6448

Company : claude
Product : claude_api_and_console
Title : "Logging in to your Console account"

If you requested the login email and clicked the link using the same device, you'll be automatically redirected to your logged-in Console account.
Clicking the link on another device
If you requested the login email and clicked the link using a different device (requesting from a web browser and clicking the email link on a phone, for example), then you will still see a "Sign in to Claude Console" button in the email, but clicking it will generate a verification code. You should enter this code 
Rank : 2
Score : 0.6385

Company : claude
Product : claude_api_and_console
Title : "Logging in to your Console account"

Check your server's quarantined emails.
Whitelist @mail.anthropic.com in your email settings.
Try logging in again after a few minutes.
**Note:** Business, government, or .edu email users may need to contact their IT department to adjust email security sett

In [ ]:
#13 — Build Context
context = pinecone_retriever.build_context(matches)

print(context[:2000])

#context = faiss_retriever.build_context(matches)

#print(context[:2000])

If you requested the login email and clicked the link using the same device, you'll be automatically redirected to your logged-in Console account.
Clicking the link on another device
If you requested the login email and clicked the link using a different device (requesting from a web browser and clicking the email link on a phone, for example), then you will still see a "Sign in to Claude Console" button in the email, but clicking it will generate a verification code. You should enter this code on the original device where you requested the login email to authenticate.
Troubleshooting
I entered my email address but I haven't received my login email.
If you requested a login email but you haven't received it yet, do the following:
Check your spam and junk folders for emails from @mail.anthropic.com.
Check your server's quarantined emails.
Whitelist @mail.anthropic.com in your email settings.
Try logging in again after a few minutes.

Check your server's quarantined emails.
Whitelist @ma